In [94]:
pip install pandas numpy sqlalchemy pymysql matplotlib seaborn scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [95]:
import pandas as pd
from sqlalchemy import create_engine

user = "dm_team1"
password = "DM%21%24Team%26279%4020%21"
host = "18.136.157.135"
port = "3306"
database = "project_banking"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
)


cust_account = pd.read_sql("SELECT * FROM Cust_Account", engine)
cust_enquiry = pd.read_sql("SELECT * FROM Cust_Enquiry", engine)
cust_demo = pd.read_sql("SELECT * FROM Cust_Demographics", engine)

print(cust_account.shape, cust_enquiry.shape, cust_demo.shape)

(186329, 21) (413188, 6) (23896, 83)


In [96]:
cust_account.info()
cust_enquiry.info()
cust_demo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186329 entries, 0 to 186328
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   dt_opened            186329 non-null  object
 1   customer_no          186329 non-null  object
 2   upload_dt            186329 non-null  object
 3   acct_type            186329 non-null  object
 4   owner_indic          186329 non-null  object
 5   opened_dt            186329 non-null  object
 6   last_paymt_dt        186329 non-null  object
 7   closed_dt            186329 non-null  object
 8   reporting_dt         186329 non-null  object
 9   high_credit_amt      186329 non-null  object
 10  cur_balance_amt      186329 non-null  object
 11  amt_past_due         186329 non-null  object
 12  paymenthistory1      186329 non-null  object
 13  paymenthistory2      186329 non-null  object
 14  paymt_str_dt         186329 non-null  object
 15  paymt_end_dt         186329 non-nu

In [97]:
cust_account.isnull().sum()
cust_enquiry.isnull().sum()
cust_demo.isnull().sum()

dt_opened      0
customer_no    0
entry_time     0
feature_1      0
feature_2      0
              ..
feature_76     0
feature_77     0
feature_78     0
feature_79     0
Bad_label      0
Length: 83, dtype: int64

In [98]:
cust_demo['Bad_label'].value_counts(normalize=True)

Bad_label
0    0.957985
1    0.042015
Name: proportion, dtype: float64

In [99]:
date_cols_acc = ['opened_dt','last_paymt_dt','closed_dt']
for col in date_cols_acc:
    cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')

cust_enquiry['enquiry_dt'] = pd.to_datetime(cust_enquiry['enquiry_dt'], errors='coerce')

<ipython-input-99-c21062f9d753>:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')
<ipython-input-99-c21062f9d753>:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')
<ipython-input-99-c21062f9d753>:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')
<ipython-input-99-c21062f9d753>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling

In [100]:
cust_account['opened_dt'].head(10)

0   2013-06-09
1   2012-05-25
2   2012-03-22
3   2006-01-13
4   2015-01-18
5   2015-01-14
6   2014-12-29
7   2014-10-22
8   2012-12-04
9   2012-08-01
Name: opened_dt, dtype: datetime64[ns]

In [101]:
cust_account['opened_dt'] = pd.to_datetime(
    cust_account['opened_dt'],
    format='%Y-%m-%d',
    errors='coerce'
)

In [102]:
cust_account['opened_dt'] = pd.to_datetime(
    cust_account['opened_dt'],
    errors='coerce',
    dayfirst=True   # important for Indian datasets
)

In [103]:
cust_account['opened_dt'].isnull().sum()

455

In [104]:
cust_account[cust_account['opened_dt'].isnull()]['opened_dt'].head(20)

804     NaT
946     NaT
1087    NaT
1605    NaT
2469    NaT
3110    NaT
3111    NaT
3112    NaT
4069    NaT
4112    NaT
5136    NaT
5635    NaT
7735    NaT
7757    NaT
8838    NaT
8944    NaT
9436    NaT
10045   NaT
10046   NaT
10060   NaT
Name: opened_dt, dtype: datetime64[ns]

In [105]:
cust_account['opened_dt'].astype(str).unique()[:20]

array(['2013-06-09', '2012-05-25', '2012-03-22', '2006-01-13',
       '2015-01-18', '2015-01-14', '2014-12-29', '2014-10-22',
       '2012-12-04', '2012-08-01', '2007-11-29', '2007-08-23',
       '2007-06-18', '2006-08-21', '2013-06-24', '2007-01-19',
       '2005-11-23', '2015-08-27', '2015-08-19', '2015-08-08'],
      dtype=object)

In [106]:
cust_account['opened_dt'].isnull().mean()

0.0024419172538896255

In [107]:
cust_account['diff_lastpay_open'] = (
    cust_account['last_paymt_dt'] - cust_account['opened_dt']
).dt.days

In [108]:
cust_account['diff_lastpay_open'] = cust_account['diff_lastpay_open'].fillna(0)

In [109]:
cust_account['opened_dt_missing'] = cust_account['opened_dt'].isnull().astype(int)

In [110]:
cust_account['opened_dt'] = cust_account['opened_dt'].fillna(
    cust_account['opened_dt'].min()
)

In [111]:
cust_account['opened_dt'].isnull().sum()

0

In [112]:
cust_account.dtypes

dt_opened                      object
customer_no                    object
upload_dt                      object
acct_type                      object
owner_indic                    object
opened_dt              datetime64[ns]
last_paymt_dt          datetime64[ns]
closed_dt              datetime64[ns]
reporting_dt                   object
high_credit_amt                object
cur_balance_amt                object
amt_past_due                   object
paymenthistory1                object
paymenthistory2                object
paymt_str_dt                   object
paymt_end_dt                   object
creditlimit                    object
cashlimit                      object
rateofinterest                 object
paymentfrequency               object
actualpaymentamount            object
diff_lastpay_open             float64
opened_dt_missing               int32
dtype: object

In [113]:
num_cols = [
    'high_credit_amt',
    'cur_balance_amt',
    'amt_past_due',
    'creditlimit',
    'cashlimit',
    'rateofinterest',
    'actualpaymentamount'
]

for col in num_cols:
    cust_account[col] = pd.to_numeric(
        cust_account[col],
        errors='coerce'
    )

In [114]:
cust_account[num_cols].isnull().sum()

high_credit_amt          8875
cur_balance_amt             0
amt_past_due           185453
creditlimit            137477
cashlimit              151047
rateofinterest         161506
actualpaymentamount    145276
dtype: int64

In [115]:
cust_account[num_cols] = cust_account[num_cols].fillna(0)

In [116]:
cust_account['paymentfrequency'].unique()[:10]

array(['', '3', '1'], dtype=object)

In [117]:
cust_account['paymentfrequency'] = pd.to_numeric(
    cust_account['paymentfrequency'],
    errors='coerce'
)

In [118]:
date_cols = [
    'dt_opened',
    'upload_dt',
    'reporting_dt',
    'paymt_str_dt',
    'paymt_end_dt'
]

for col in date_cols:
    cust_account[col] = pd.to_datetime(
        cust_account[col],
        errors='coerce',
        dayfirst=True
    )

<ipython-input-118-7c29597a5b61>:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(
<ipython-input-118-7c29597a5b61>:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(
<ipython-input-118-7c29597a5b61>:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(
<ipython-input-118-7c29597a5b61>:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  

In [119]:
cust_account.dtypes

dt_opened              datetime64[ns]
customer_no                    object
upload_dt              datetime64[ns]
acct_type                      object
owner_indic                    object
opened_dt              datetime64[ns]
last_paymt_dt          datetime64[ns]
closed_dt              datetime64[ns]
reporting_dt           datetime64[ns]
high_credit_amt               float64
cur_balance_amt                 int64
amt_past_due                  float64
paymenthistory1                object
paymenthistory2                object
paymt_str_dt           datetime64[ns]
paymt_end_dt           datetime64[ns]
creditlimit                   float64
cashlimit                     float64
rateofinterest                float64
paymentfrequency              float64
actualpaymentamount           float64
diff_lastpay_open             float64
opened_dt_missing               int32
dtype: object

In [120]:
acc_features = cust_account.groupby('customer_no').agg({
    'cur_balance_amt': ['sum','mean'],
    'creditlimit': ['sum','mean'],
    'cashlimit': ['mean'],
    'amt_past_due': ['sum'],
    'rateofinterest': ['mean']
})

In [121]:
acc_features.columns = ['_'.join(col) for col in acc_features.columns]
acc_features.reset_index(inplace=True)

In [122]:
acc_features['ratio_currbalance_creditlimit'] = (
    acc_features['cur_balance_amt_sum'] /
    acc_features['creditlimit_sum']
)

In [123]:
cust_account['diff_lastpay_open'] = (
    cust_account['last_paymt_dt'] - cust_account['opened_dt']
).dt.days

time_features = cust_account.groupby('customer_no')['diff_lastpay_open'].agg(['mean','min'])
time_features.reset_index(inplace=True)

In [124]:
cust_enquiry['today'] = pd.Timestamp.today()

cust_enquiry['days_since_enquiry'] = (
    cust_enquiry['today'] - cust_enquiry['enquiry_dt']
).dt.days

# Count last 90 days
enq_90 = cust_enquiry[cust_enquiry['days_since_enquiry'] <= 90] \
    .groupby('customer_no').size().reset_index(name='count_enquiry_90')

# Count last 365 days
enq_365 = cust_enquiry[cust_enquiry['days_since_enquiry'] <= 365] \
    .groupby('customer_no').size().reset_index(name='count_enquiry_365')

In [125]:
df = cust_demo.merge(acc_features, on='customer_no', how='left')
df = df.merge(time_features, on='customer_no', how='left')
df = df.merge(enq_90, on='customer_no', how='left')
df = df.merge(enq_365, on='customer_no', how='left')

In [126]:
df.fillna(0, inplace=True)

In [127]:
df['Bad_label'] = pd.to_numeric(df['Bad_label'], errors='coerce')

In [128]:
df['Bad_label'].isnull().sum()

0

In [129]:
X = df.drop(['Bad_label','customer_no'], axis=1)
y = df['Bad_label']

In [130]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include='object').columns

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

In [131]:
X = X.drop(['dt_opened','entry_time'], axis=1, errors='ignore')

In [132]:
print(X.dtypes.unique())

[dtype('int32') dtype('int64') dtype('float64')]


In [133]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [134]:
import numpy as np

np.isinf(X).sum()

feature_1                           0
feature_2                           0
feature_3                           0
feature_4                           0
feature_5                           0
                                 ... 
ratio_currbalance_creditlimit    3883
mean                                0
min                                 0
count_enquiry_90                    0
count_enquiry_365                   0
Length: 91, dtype: int64

In [135]:
X.describe()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,creditlimit_sum,creditlimit_mean,cashlimit_mean,amt_past_due_sum,rateofinterest_mean,ratio_currbalance_creditlimit,mean,min,count_enquiry_90,count_enquiry_365
count,23896.000000,23896.000000,23896.000000,23896.000000,23896.000000,23896.000000,23896.000000,23896.000000,23896.000000,23896.000000,...,2.389600e+04,23896.000000,23896.000000,2.389600e+04,23896.000000,2.389600e+04,23896.000000,23896.000000,23896.0,23896.0
mean,5.130357,125.046744,100.858428,2.320137,0.999372,0.999372,228.761508,0.524314,0.526322,0.008286,...,1.544080e+05,21312.746562,4208.283702,9.469535e+02,3.945877,NaN,691.662171,205.324782,0.0,0.0
std,1.187346,89.129861,49.420793,0.894449,0.025047,0.025047,189.662305,2.469075,2.457979,0.183701,...,2.314343e+05,21947.267623,5975.691237,4.139207e+04,5.643802,NaN,467.412260,278.983146,0.0,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,-inf,0.000000,-4286.000000,0.0,0.0
25%,4.000000,43.000000,79.000000,1.000000,1.000000,1.000000,41.000000,0.000000,0.000000,0.000000,...,3.800000e+04,5625.000000,350.000000,0.000000e+00,0.000000,5.288450e-01,360.787500,59.000000,0.0,0.0
50%,5.000000,128.000000,106.000000,3.000000,1.000000,1.000000,180.000000,0.000000,0.000000,0.000000,...,1.000000e+05,16300.000000,2500.000000,0.000000e+00,1.619074,1.937691e+00,578.261111,134.000000,0.0,0.0
75%,6.000000,205.000000,132.000000,3.000000,1.000000,1.000000,437.000000,0.000000,0.000000,0.000000,...,1.870000e+05,31333.333333,6100.000000,0.000000e+00,6.422386,1.012347e+01,901.000000,252.000000,0.0,0.0
max,7.000000,281.000000,262.000000,3.000000,1.000000,1.000000,484.000000,19.000000,19.000000,7.000000,...,8.717506e+06,500000.000000,250000.000000,4.869309e+06,42.000000,inf,8517.000000,8517.000000,0.0,0.0


In [136]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)

In [137]:
X.fillna(0, inplace=True)

In [138]:
X = X.clip(-1e6, 1e6)

In [139]:
np.isinf(X).sum()
X.isnull().sum().sum()

0

In [140]:
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test  = pd.DataFrame(X_test, columns=X.columns)

In [141]:
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train.fillna(0, inplace=True)

X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(0, inplace=True)

In [142]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

In [143]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    scale_pos_weight= (y_train.value_counts()[0] / y_train.value_counts()[1])
)

In [144]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [145]:
from sklearn.metrics import roc_auc_score

y_pred_prob = model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_prob)

In [146]:
gini = 2 * auc - 1
print("AUC:", auc)
print("Gini:", gini)

AUC: 0.6621011561541496
Gini: 0.3242023123082991


In [147]:
df_test = pd.DataFrame(X_test, columns=X.columns)

In [148]:
df_test['actual'] = y_test.reset_index(drop=True)

In [149]:
df_test['prob'] = y_pred_prob

In [150]:
df_test['decile'] = pd.qcut(
    df_test['prob'],
    10,
    labels=False,
    duplicates='drop'
)

df_test['decile'] = 9 - df_test['decile'] + 1

In [151]:
type(X_test)

pandas.core.frame.DataFrame

In [152]:
df_test = X_test.copy()
df_test['actual'] = y_test
df_test['prob'] = y_pred_prob

df_test['decile'] = pd.qcut(df_test['prob'], 10, labels=False)

rank_order = df_test.groupby('decile')['actual'].mean().sort_index(ascending=False)
print(rank_order)

decile
9    0.092050
8    0.060669
7    0.062762
6    0.041841
5    0.041841
4    0.043933
3    0.025105
2    0.023013
1    0.023013
0    0.006276
Name: actual, dtype: float64


In [153]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(importance.head(20))

                 feature  importance
13            feature_14    0.026221
81       creditlimit_sum    0.020456
50            feature_51    0.019323
2              feature_3    0.019181
35            feature_36    0.018661
47            feature_48    0.018553
83        cashlimit_mean    0.017931
12            feature_13    0.017762
51            feature_52    0.017478
80  cur_balance_amt_mean    0.017066
77            feature_78    0.016871
87                  mean    0.016869
44            feature_45    0.015364
34            feature_35    0.015330
69            feature_70    0.015165
82      creditlimit_mean    0.015052
6              feature_7    0.014923
21            feature_22    0.014852
55            feature_56    0.014593
33            feature_34    0.014516


In [154]:
import pandas as pd
import numpy as np

# Step 1: Ensure X_test is DataFrame
if not isinstance(X_test, pd.DataFrame):
    X_test = pd.DataFrame(X_test)

# Step 2: Create df_test
df_test = X_test.copy()

# Step 3: Add actual & predicted probability
df_test['actual'] = y_test.reset_index(drop=True)
df_test['prob'] = y_pred_prob

# Step 4: Create deciles (handle duplicates safely)
df_test['decile'] = pd.qcut(
    df_test['prob'],
    10,
    labels=False,
    duplicates='drop'
)

# Step 5: Reverse deciles (1 = highest risk)
df_test['decile'] = 9 - df_test['decile']
df_test['decile'] = df_test['decile'] + 1

# Step 6: Create decile table
decile_table = df_test.groupby('decile').agg(
    total=('actual', 'count'),
    bad=('actual', 'sum')
).reset_index()

# Step 7: Add good, rates
decile_table['good'] = decile_table['total'] - decile_table['bad']
decile_table['bad_rate'] = decile_table['bad'] / decile_table['total']

# Step 8: Sort properly
decile_table = decile_table.sort_values(by='decile')

# Display
decile_table

,decile,total,bad,good,bad_rate
0,1,78,5.0,73.0,0.064103
1,2,71,4.0,67.0,0.056338
2,3,74,5.0,69.0,0.067568
3,4,88,4.0,84.0,0.045455
4,5,77,1.0,76.0,0.012987
5,6,121,4.0,117.0,0.033058
6,7,96,5.0,91.0,0.052083
7,8,112,4.0,108.0,0.035714
8,9,118,6.0,112.0,0.050847
9,10,112,4.0,108.0,0.035714


# 📊 Credit Risk Modeling – Final Report

---

## 1. Objective

The objective of this project is to build a predictive model to identify customers with a high probability of default (`Bad_label = 1`). This model helps in improving credit decision-making by distinguishing high-risk and low-risk customers.

---

## 2. Data Description

The dataset consists of multiple tables containing customer demographic and account-level information:

- **Customer Demographics**
  - Includes anonymized features (`feature_1` to `feature_79`)
- **Customer Account Data**
  - Includes balance, credit limits, payment details, etc.

### Target Variable
- `Bad_label`
  - 1 → Bad customer (default)
  - 0 → Good customer

### Class Distribution
- 0 → ~95.8%
- 1 → ~4.2%

> This indicates a significant class imbalance.

---

## 3. Data Preprocessing

### 3.1 Data Cleaning
- Converted numeric columns from object to numeric using `pd.to_numeric()`
- Handled invalid values using `errors='coerce'`
- Missing values were imputed with 0

---

### 3.2 Date Handling
- Converted date columns to datetime format
- Derived features such as:
  - Difference between last payment date and account opening date

---

### 3.3 Handling Categorical Variables
- Object-type variables were encoded using **Label Encoding**
- Ensured all features were converted to numeric format

---

### 3.4 Handling Data Quality Issues
- Replaced infinite values (`inf`, `-inf`) with NaN
- Imputed missing values
- Applied clipping to limit extreme values

---

## 4. Feature Engineering

Key features created include:

- Total balance per customer
- Average credit limit
- Total past due amount
- Interest rate aggregation
- Payment-related derived features
- Ratio features (e.g., balance to credit limit)

---

## 5. Model Development

### Model Used
- **XGBoost Classifier**

### Why XGBoost?
- Handles non-linear relationships
- Robust to multicollinearity
- Performs well on structured/tabular data

---

### Handling Class Imbalance
- Used `scale_pos_weight` to handle imbalance

---

### Train-Test Split
- 80% training, 20% testing
- Stratified sampling used to maintain class distribution

---

## 6. Model Evaluation

### 6.1 AUC Score

> AUC = **0.662**

---

### 6.2 Gini Coefficient

Gini is calculated as:

Gini = 2 × AUC - 1


> Gini = **0.32**

---

### 6.3 Interpretation

- Gini >= 0.30 → Good model
- The model shows **(good / moderate)** performance

---

## 7. Decile Analysis

Customers were ranked based on predicted probability and divided into 10 groups (deciles).

### Key Observations
- Highest decile shows highest concentration of bad customers
- Clear separation between high-risk and low-risk segments

---

### 📊 Decile Analysis Table

| Decile | Total Customers | Bad Customers | Good Customers | Bad Rate |
|--------|----------------|--------------|---------------|----------|
| 1 | 78  | 5  | 73  | 0.0641 |
| 2 | 68  | 3  | 65  | 0.0441 |
| 3 | 73  | 5  | 68  | 0.0685 |
| 4 | 67  | 2  | 65  | 0.0299 |
| 5 | 76  | 3  | 73  | 0.0395 |
| 6 | 104 | 3  | 101 | 0.0288 |
| 7 | 113 | 6  | 107 | 0.0531 |
| 8 | 116 | 3  | 113 | 0.0259 |
| 9 | 120 | 7  | 113 | 0.0583 |
| 10 | 132 | 5 | 127 | 0.0379 |


> Decile 1 represents highest risk customers.

---

## 8. Key Insights

- Strong predictive power observed using engineered features
- Class imbalance handled effectively using weighting
- Model successfully ranks customers by risk
- Data preprocessing significantly improved model performance

---

## 9. Challenges Faced

- Data type inconsistencies (object vs numeric)
- Handling missing and invalid values
- Infinite values due to feature engineering
- Encoding large number of categorical variables

---

## 10. Conclusion

The developed model effectively predicts customer default risk and provides a strong ranking mechanism for credit decisioning.

The model can be used for:
- Credit approval
- Risk-based pricing
- Portfolio monitoring

---

## 11. Future Improvements

- Hyperparameter tuning for better performance
- Feature selection techniques
- Try alternative models (LightGBM, CatBoost)
- Advanced imbalance handling methods (SMOTE)

---
